# 02 · Block size and spatial folds (blockCV in R)

Only Valavi's **blockCV** package decides the block size and the folds; nothing of it is re-written.

| Step | blockCV function | What it gives |
|---|---|---|
| Install | `remotes::install_github("rvalavi/blockCV@51e6903…")` | the GitHub version 4.0-2, pinned to one commit |
| Block size | `cv_spatial_autocor(x = boreholes, column = 5 slices of 3 m)` | one range per slice; **blockCV's `$range` = their median = the block size** |
| Folds | `cv_spatial(size = block, k = 5, hexagon = FALSE)` | 5 folds of square blocks, also for 2× and 3× the block size |

Each borehole is one point. Its columns are its clay share at 0-3, 3-6, 6-9, 9-12 and 12-15 m. Boreholes without cells in every slice are left out of the block-size step only (blockCV cannot use empty values). The other slice thicknesses (1.5, 5, 15 m) are run **for information only**; they do not change the block size.

**Before you run:** set `RSCRIPT` and `RTOOLS` below (the defaults are the school PC's). The first run installs blockCV from GitHub (needs internet and Rtools, 5-10 min).

**Run time:** 2-5 minutes after the first install.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import glob, os, shutil, subprocess

RSCRIPT = r"E:\DavidAn\R\R-4.6.1\bin\Rscript.exe"   # full path of Rscript.exe (None = search)
RTOOLS = r"E:\DavidAn\rtools46"                     # Rtools folder (Windows)

if RTOOLS and os.path.isdir(RTOOLS):                 # let R find Rtools to compile blockCV
    for v in ("44", "45", "46", "47"):
        os.environ[f"RTOOLS{v}_HOME"] = RTOOLS
    os.environ["PATH"] = os.pathsep.join([os.path.join(RTOOLS, "usr", "bin"),
                                          os.path.join(RTOOLS, "x86_64-w64-mingw32.static.posix", "bin"), os.environ["PATH"]])

def find_rscript():
    if RSCRIPT and os.path.exists(RSCRIPT):
        return RSCRIPT
    found = shutil.which("Rscript") or sorted(glob.glob(r"C:\Program Files\R\R-*\bin\Rscript.exe"))
    return found if isinstance(found, str) else (found[-1] if found else None)

def run_r(script):
    """Run one R script and show its messages while it runs."""
    rscript = find_rscript()
    if rscript is None:
        raise FileNotFoundError("Rscript not found: set RSCRIPT above")
    print(">>", script, flush=True)
    with subprocess.Popen([rscript, script], cwd=S["root"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as proc:
        for line in proc.stdout:
            if "suspicious field assignments" not in line and not line.startswith("code for methods"):
                print(line, end="", flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"{script} failed (see the messages above)")

H.write_r_settings(S)
run_r("R/01_install_blockcv.R")

## 1. Block size and folds

In [ ]:
run_r("R/02_blockcv.R")

## 2. Results

In [ ]:
OUT = H.out_dir(S, "02_blockcv")
dec = json.loads((H.folds_dir(S) / "block_size.json").read_text())
print(f"Block size = {dec['block_m']:.1f} m  (blockCV {dec['blockcv_version']}, commit {dec['blockcv_commit']}, "
      f"{dec['boreholes_used']} boreholes, {len(dec['slices'])} slices of {dec['slice_m']:g} m)")
print("blockCV range of each slice (m):", {k: round(v, 1) for k, v in dec["range_by_slice_m"].items()})
print("\nOther slice thicknesses - information only, not used:")
display(pd.read_csv(OUT / "range_other_slices_info_only.csv").round(1))
display(Image(str(OUT / "fig_range_by_slice.png"), width=450))

In [ ]:
fs = pd.read_csv(OUT / "fold_summary.csv")
display(fs.pivot_table(index="fold", columns="block_m", values=["test_boreholes", "test_clay_share"]))
display(Image(str(H.out_dir(S, "02_blockcv", private=True) / "fig_folds_x1.png"), width=600))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 03.")